# 3.2 Arquitectura del Data Warehouse: El Esquema de Estrella (Banca)
Vamos a simular un Data Warehouse en memoria. Tenemos una **Tabla de Hechos** (las transacciones crudas) rodeada de **Tablas de Dimensión** (el contexto de la transacción). Nuestro objetivo de KDD es crear una matriz plana para alimentar a un modelo de riesgo.

In [ ]:
import pandas as pd

print("--- 1. TABLAS DE DIMENSIÓN (El Contexto) ---")
# Dimensión Cliente (Desnormalizada: el segmento ya viene escrito, no es otro ID)
dim_clientes = pd.DataFrame({
    'id_cliente': [101, 102, 103, 104],
    'estado': ['BC', 'CDMX', 'BC', 'Jalisco'],
    'segmento': ['Universitario', 'Premium', 'Universitario', 'Standard']
})

display(dim_clientes)

# Dimensión Tiempo
dim_tiempo = pd.DataFrame({
    'id_tiempo': [20260914, 20260915],
    'dia_semana': ['Lunes', 'Martes'],
    'es_quincena': [False, True]
})
display(dim_tiempo)

print("\n--- 2. TABLA DE HECHOS (La Métrica) ---")
# La tabla de hechos solo une IDs numéricos y la métrica (Monto)
fact_transacciones = pd.DataFrame({
    'id_transaccion': [1, 2, 3, 4, 5],
    'id_cliente': [101, 102, 101, 104, 103],
    'id_tiempo': [20260914, 20260914, 20260915, 20260915, 20260915],
    'monto_mxn': [150.50, 4500.00, 85.00, 1200.00, 300.00]
})


display(fact_transacciones)

: 

### Ejecutando la Tubería ETL en Memoria
Para que el modelo de Machine Learning entienda esto, debemos pasar por las tres fases de la arquitectura de datos.

In [ ]:
print("--- INICIANDO PIPELINE ETL ---")

# EXTRACT & TRANSFORM: Cruzamos los hechos con las dimensiones (Desnormalización masiva)
matriz_kdd = pd.merge(fact_transacciones, dim_clientes, on='id_cliente', how='inner')
matriz_kdd = pd.merge(matriz_kdd, dim_tiempo, on='id_tiempo', how='inner')

# TRANSFORM: Purgamos llaves primarias operativas que causan ruido matemático
matriz_kdd = matriz_kdd.drop(columns=['id_transaccion', 'id_cliente', 'id_tiempo'])

# LOAD: La matriz final se "carga" en la variable de memoria lista para la IA
print("--- MATRIZ MULTIDIMENSIONAL (LOAD COMPLETO) ---")
display(matriz_kdd)

---
### 🛠️ Zona de Experimentación y Reto de Código

En los Almacenes de Datos, la operación analítica clásica se llama **Roll-up** (Acumular o Resumir datos subiendo de granularidad). 

**Reto 1: Código Analítico**
Usa Pandas para responder a la pregunta que tiró el servidor de producción: *¿Cuál es el monto total de dinero transaccionado por cada Segmento de Cliente?*
Escribe el código usando la función `.groupby()` sobre tu `matriz_kdd`.

In [ ]:
# ESCRIBE TU CÓDIGO AQUÍ:
# Pista: Agrupa por 'segmento', selecciona 'monto_mxn' y aplica .sum()

resultado_rollup = matriz_kdd.groupby('segmento')['monto_mxn'].sum()
print(resultado_rollup)

**Reto 2: Arquitectura en Papel (Responde aquí mismo)**
Observa nuestra tabla `dim_clientes`. En una base de datos relacional estricta (OLTP), el segmento "Universitario" sería un número (un ID_Segmento) que apuntaría a otra tabla de catálogo. 
En el Almacén de Datos (OLAP) decidimos escribir "Universitario" directamente en texto, repitiéndolo por cada cliente. 
Como ingeniero de software: ¿Por qué estamos violando las reglas de normalización a propósito en esta arquitectura? ¿Qué ganamos al aceptar esa redundancia?

> **Tu respuesta:** Como ing. de software, desnormalizamos para hacer las consultas más rápidas y sencillas. Aunque se repita información, podemos analizar y agrupar los datos sin realizar tantos cruces entre tablas.